In [1]:
# ============================================================================
# CROP YIELD PREDICTION - ML ENGINEERING PIPELINE
# ============================================================================
# ML Protocols: Data Preprocessing → Feature Engineering → Model Training
#               Model Evaluation → Statistical Validation → Production Ready
# ============================================================================

import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy import stats
from xgboost import XGBRegressor
import joblib
import warnings

warnings.filterwarnings('ignore')

# Define missing variables
X_train, X_test, y_train, y_test = [], [], [], []

print("✅ ML Pipeline Initialized - All libraries loaded successfully")

✅ ML Pipeline Initialized - All libraries loaded successfully


In [2]:
# ============================================================================
# PHASE 1: DATA LOADING & INITIAL EXPLORATION
# ============================================================================

# Load datasets
crops = pd.read_csv("../data/final provincial dataset.csv")
weather = pd.read_csv("../data/province_yearly_weather_final.csv")

# Standardize column names
crops.columns = crops.columns.str.lower().str.strip()
weather.columns = weather.columns.str.lower().str.strip()

print("📊 Dataset Loading Complete")
print(f"\nCrops Dataset Shape: {crops.shape}")
print(f"Weather Dataset Shape: {weather.shape}")
print(f"\nCrops Columns: {crops.columns.tolist()}")
print(f"Weather Columns: {weather.columns.tolist()}")

# Display basic statistics
print(f"\n🌾 Unique Provinces: {crops['province'].unique().tolist()}")
print(f"🌾 Unique Crops: {crops['crop'].unique().tolist()}")
print(f"📅 Year Range: {crops['year'].min()}-{crops['year'].max()}")


📊 Dataset Loading Complete

Crops Dataset Shape: (1232, 6)
Weather Dataset Shape: (100, 7)

Crops Columns: ['year', 'crop', 'province', 'area_acres', 'production_tonnes', 'yield_tonnes_per_acre']
Weather Columns: ['province', 'year', 'avg_temp', 'min_temp', 'max_temp', 'rainfall', 'humidity']

🌾 Unique Provinces: ['Punjab', 'Sindh', 'KPK', 'Balochistan', 'KPP']
🌾 Unique Crops: ['Wheat', 'Rice', 'Maize', 'Cotton']
📅 Year Range: 1948-2024


In [3]:
# ============================================================================
# PHASE 2: DATA INTEGRATION & PREPROCESSING
# ============================================================================

# Merge datasets on province and year
df = pd.merge(
    crops,
    weather,
    on=["province", "year"],
    how="inner"
)

print(f"✅ Data Merged: {df.shape[0]} records")
print(f"\n📋 Initial Missing Values:\n{df.isnull().sum()}")

# Statistical check - Data quality assessment
print(f"\n📊 Missing Data Analysis:")
print(f"Missing Percentage: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%")


✅ Data Merged: 400 records

📋 Initial Missing Values:
year                     0
crop                     0
province                 0
area_acres               0
production_tonnes        0
yield_tonnes_per_acre    0
avg_temp                 0
min_temp                 0
max_temp                 0
rainfall                 0
humidity                 0
dtype: int64

📊 Missing Data Analysis:
Missing Percentage: 0.00%


In [4]:
# Separate features and target
X = df.drop(columns=["yield_tonnes_per_acre"])
y = df["yield_tonnes_per_acre"]

# Identify column types
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

# Feature selection pipeline
selector = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("select", SelectKBest(score_func=f_regression, k=15))  # choose top 15
    ]
)

# Fit
selector.fit(X, y)

# Get feature names after encoding
feature_names = selector.named_steps["preprocess"].get_feature_names_out()

# Get selected mask
mask = selector.named_steps["select"].get_support()

# Selected feature names
selected_features = feature_names[mask]

print("Selected Features:")
for f in selected_features:
    print(f)


Selected Features:
num__year
num__area_acres
num__production_tonnes
num__avg_temp
num__min_temp
num__max_temp
num__rainfall
num__humidity
cat__crop_Cotton
cat__crop_Maize
cat__crop_Rice
cat__crop_Wheat
cat__province_Balochistan
cat__province_KPK
cat__province_Punjab


In [5]:
# ============================================================================
# PHASE 7: TRAIN-TEST SPLIT
# ============================================================================

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print(f"✅ Data Split Complete:")
print(f"   Training samples: {X_train.shape[0]}")
print(f"   Testing samples: {X_test.shape[0]}")
print(f"   Features: {X_train.shape[1]}")


✅ Data Split Complete:
   Training samples: 320
   Testing samples: 80
   Features: 10


In [6]:
# =============================================================
# REFINED CELL 8: UNIFIED PRODUCTION PIPELINE
# =============================================================

# 1. Define our columns using actual dataframe columns
# Use the columns identified earlier (num_features and cat_features)
num_cols = num_features.tolist()
cat_cols = cat_features.tolist()

# 2. Create the "Sorting Machine" (Preprocessor)
# This scales the numbers and encodes the words automatically
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# 3. Create the "Full Lego Set" (The Pipeline)
# This snaps the Sorting Machine and the XGBoost Brain together
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=100, 
        learning_rate=0.1, 
        max_depth=5, 
        random_state=42
    ))
])

# 4. Train the whole pipeline at once
# We use the raw X_train (not pre-scaled) because the pipeline does it for us!
model.fit(X_train, y_train)

print("✅ SUCCESS: Production Pipeline Created and Trained!")
print("This model now 'wears a hat' that handles all data scaling automatically.")

✅ SUCCESS: Production Pipeline Created and Trained!
This model now 'wears a hat' that handles all data scaling automatically.


In [7]:
# ============================================================================
# PHASE 8: MODEL EVALUATION & STATISTICAL VALIDATION
# ============================================================================

# Generate predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# ============================================================================
# REGRESSION METRICS COMPUTATION
# ============================================================================

# Training Set Metrics
train_r2 = r2_score(y_train, y_pred_train)
train_mae = mean_absolute_error(y_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_mape = np.mean(np.abs((y_train - y_pred_train) / y_train)) * 100

# Test Set Metrics
test_r2 = r2_score(y_test, y_pred_test)
test_mae = mean_absolute_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mape = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100

# ============================================================================
# ACCURACY CALCULATION (Classification-style accuracy for regression)
# ============================================================================

# Calculate accuracy as percentage of predictions within acceptable tolerance
tolerance = test_rmse  # Use RMSE as tolerance threshold
train_accuracy = np.mean(np.abs(y_train - y_pred_train) <= tolerance) * 100
test_accuracy = np.mean(np.abs(y_test - y_pred_test) <= tolerance) * 100

# Alternative accuracy: correlation-based accuracy
train_correlation_accuracy = np.corrcoef(y_train, y_pred_train)[0, 1] * 100
test_correlation_accuracy = np.corrcoef(y_test, y_pred_test)[0, 1] * 100

print("=" * 70)
print("📊 MODEL PERFORMANCE METRICS")
print("=" * 70)

print(f"\n📈 TRAINING SET:")
print(f"   R² Score:  {train_r2:.4f}  (Explains {train_r2*100:.2f}% variance)")
print(f"   Accuracy:  {train_accuracy:.2f}%  (within {tolerance:.4f} tonnes/acre)")
print(f"   Correlation Accuracy: {train_correlation_accuracy:.2f}%")
print(f"   MAE:       {train_mae:.4f}  tonnes/acre")
print(f"   RMSE:      {train_rmse:.4f}  tonnes/acre")
print(f"   MAPE:      {train_mape:.2f}%")

print(f"\n🎯 TEST SET (Validation):")
print(f"   R² Score:  {test_r2:.4f}  (Explains {test_r2*100:.2f}% variance)")
print(f"   Accuracy:  {test_accuracy:.2f}%  (within {test_rmse:.4f} tonnes/acre)")
print(f"   Correlation Accuracy: {test_correlation_accuracy:.2f}%")
print(f"   MAE:       {test_mae:.4f}  tonnes/acre")
print(f"   RMSE:      {test_rmse:.4f}  tonnes/acre")
print(f"   MAPE:      {test_mape:.2f}%")

# Overfitting Analysis
overfitting_gap = train_r2 - test_r2
print(f"\n⚠️  Overfitting Analysis:")
print(f"   R² Gap (Train-Test): {overfitting_gap:.4f}")

if overfitting_gap < 0.05:
    print(f"   Status: ✅ Well-generalized (gap < 0.05)")
elif overfitting_gap < 0.10:
    print(f"   Status: ⚠️  Slight overfitting (gap < 0.10)")
else:
    print(f"   Status: ❌ Significant overfitting detected")

# ============================================================================
# RESIDUAL ANALYSIS
# ============================================================================

residuals = y_test - y_pred_test

print(f"\n📊 Residual Analysis (Test Set):")
print(f"   Mean Residual:     {residuals.mean():.6f}  (should be ~0)")
print(f"   Std Dev Residual:  {residuals.std():.4f}")
print(f"   Min Residual:      {residuals.min():.4f}")
print(f"   Max Residual:      {residuals.max():.4f}")

# Normality test (Shapiro-Wilk)
stat, p_value = stats.shapiro(residuals)

print(f"\n   Shapiro-Wilk Test:")
print(f"   p-value: {p_value:.4f}")
if p_value > 0.05:
    print(f"   ✅ Residuals normally distributed")
else:
    print(f"   ⚠️  Residuals may not be normally distributed")

print("=" * 70)


📊 MODEL PERFORMANCE METRICS

📈 TRAINING SET:
   R² Score:  0.9997  (Explains 99.97% variance)
   Accuracy:  100.00%  (within 0.0817 tonnes/acre)
   Correlation Accuracy: 99.98%
   MAE:       0.0072  tonnes/acre
   RMSE:      0.0098  tonnes/acre
   MAPE:      1.48%

🎯 TEST SET (Validation):
   R² Score:  0.9814  (Explains 98.14% variance)
   Accuracy:  83.75%  (within 0.0817 tonnes/acre)
   Correlation Accuracy: 99.07%
   MAE:       0.0507  tonnes/acre
   RMSE:      0.0817  tonnes/acre
   MAPE:      9.01%

⚠️  Overfitting Analysis:
   R² Gap (Train-Test): 0.0183
   Status: ✅ Well-generalized (gap < 0.05)

📊 Residual Analysis (Test Set):
   Mean Residual:     -0.002622  (should be ~0)
   Std Dev Residual:  0.0822
   Min Residual:      -0.4499
   Max Residual:      0.2119

   Shapiro-Wilk Test:
   p-value: 0.0000
   ⚠️  Residuals may not be normally distributed


In [ ]:
# ============================================================================
# PHASE 9: FEATURE IMPORTANCE ANALYSIS
# ============================================================================

# Get feature importance from XGBoost model
xgb_model = model.named_steps['regressor']
importance_dict = xgb_model.get_booster().get_score(importance_type='weight')

# Map importance values to features, using 0 for missing features
importance_values = [importance_dict.get(f, 0) for f in X.columns]
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': importance_values
}).sort_values('importance', ascending=False)

# Normalize importance
feature_importance['importance_pct'] = (feature_importance['importance'] / feature_importance['importance'].sum()) * 100

print("🎯 Feature Importance Ranking:")
print(feature_importance.to_string(index=False))

# Plot feature importance
fig = go.Figure()
fig.add_trace(go.Bar(
    x=feature_importance['importance_pct'],
    y=feature_importance['feature'],
    orientation='h',
    marker_color='rgba(55, 128, 191, 0.7)',
    text=feature_importance['importance_pct'].round(2),
    textposition='auto'
))
fig.update_layout(
    title="Feature Importance in XGBoost Model (%)",
    xaxis_title="Importance Percentage",
    yaxis_title="Features",
    height=400,
    showlegend=False
)
fig.write_html("02_feature_importance.html")
print("\n✅ Saved: 02_feature_importance.html")


🎯 Feature Importance Ranking:
          feature  importance  importance_pct
             year           0             NaN
             crop           0             NaN
         province           0             NaN
       area_acres           0             NaN
production_tonnes           0             NaN
         avg_temp           0             NaN
         min_temp           0             NaN
         max_temp           0             NaN
         rainfall           0             NaN
         humidity           0             NaN

✅ Saved: 02_feature_importance.html


In [9]:
# =============================================================
# REFINED CELL 10: PACKING THE SUITCASE (SAVING THE PIPELINE)
# =============================================================

# 1. Create the models folder if it doesn't exist yet
if not os.path.exists('../models'):
    os.makedirs('../models')

# 2. Save the ENTIRE pipeline (preprocessor + XGBoost)
# We use compress=1 to keep everything in ONE file
model_filename = '../models/xgb_model.pkl'
joblib.dump(model, model_filename, compress=1)

print(f"✅ SUCCESS: Pipeline saved as '{model_filename}'")
print("📁 Model saved to: models/xgb_model.pkl")

# 3. Quick Test: Try loading it back and predicting one row
loaded_model = joblib.load(model_filename)
test_pred = loaded_model.predict(X_test.iloc[[0]])
print(f"🔬 Test Prediction on first row: {test_pred[0]:.4f} tonnes/acre")

✅ SUCCESS: Pipeline saved as '../models/xgb_model.pkl'
📁 Model saved to: models/xgb_model.pkl
🔬 Test Prediction on first row: 1.4383 tonnes/acre


In [10]:
# ============================================================================
# PHASE 10B: SIMPLE PREDICTION FUNCTION
# ============================================================================

def predict_yield_simple(X_features):
    """
    Predict crop yield using the trained model
    
    Parameters:
    - X_features: DataFrame or array-like with the same features as X
    
    Returns:
    - array with predictions
    """
    try:
        # Load model from models folder
        loaded_model = joblib.load("../models/xgb_model.pkl")
        
        # Make prediction
        prediction = loaded_model.predict(X_features)
        
        return prediction
    
    except Exception as e:
        return {"error": str(e)}

# Test the prediction function with a test sample
print("✅ Prediction Function Ready")
print(f"\n📋 Test Prediction on first test sample:")
test_sample = X_test.iloc[0:1]
test_pred = predict_yield_simple(test_sample)
print(f"Predicted Yield: {test_pred[0]:.4f} tonnes/acre")
print(f"Actual Yield: {y_test.iloc[0]:.4f} tonnes/acre")
print(f"Error: {abs(test_pred[0] - y_test.iloc[0]):.4f} tonnes/acre")


✅ Prediction Function Ready

📋 Test Prediction on first test sample:
Predicted Yield: 1.4383 tonnes/acre
Actual Yield: 1.3760 tonnes/acre
Error: 0.0623 tonnes/acre


In [11]:
# ============================================================================
# PHASE 12: COMPREHENSIVE VISUALIZATION & ANALYSIS
# ============================================================================

# Set output directory for reports
output_dir = '../frontend/static/reports'
os.makedirs(output_dir, exist_ok=True)

# ============================================================================
# 1. MODEL DIAGNOSTICS
# ============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Actual vs Predicted (Test)", "Residuals Distribution", 
                    "Residuals vs Predicted", "Q-Q Plot")
)

# Plot 1: Actual vs Predicted
fig.add_trace(go.Scatter(x=y_test, y=y_pred_test, mode='markers', 
                         name='Predictions', marker=dict(size=6, color='blue', opacity=0.6)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
                         mode='lines', name='Perfect Prediction', 
                         line=dict(color='red', dash='dash')),
              row=1, col=1)

# Plot 2: Residuals Histogram
fig.add_trace(go.Histogram(x=residuals, name='Residuals', nbinsx=30, 
                          marker_color='green', opacity=0.7),
              row=1, col=2)

# Plot 3: Residuals vs Predicted
fig.add_trace(go.Scatter(x=y_pred_test, y=residuals, mode='markers',
                        name='Residuals', marker=dict(size=6, color='purple', opacity=0.6)),
              row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=1)

# Plot 4: Q-Q Plot
sorted_residuals = np.sort(residuals)
theoretical_quantiles = stats.norm.ppf(np.arange(1, len(residuals)+1) / (len(residuals)+1))
fig.add_trace(go.Scatter(x=theoretical_quantiles, y=sorted_residuals, mode='markers',
                        name='Q-Q', marker=dict(size=5, color='orange')),
              row=2, col=2)

fig.update_xaxes(title_text="Actual Yield", row=1, col=1)
fig.update_yaxes(title_text="Predicted Yield", row=1, col=1)
fig.update_xaxes(title_text="Residuals", row=1, col=2)
fig.update_xaxes(title_text="Predicted Yield", row=2, col=1)
fig.update_yaxes(title_text="Residuals", row=2, col=1)
fig.update_xaxes(title_text="Theoretical Quantiles", row=2, col=2)
fig.update_yaxes(title_text="Sample Quantiles", row=2, col=2)

fig.update_layout(height=800, showlegend=True, title_text="Model Diagnostics - Test Set")
fig.write_html(f"{output_dir}/01_model_diagnostics.html")
print("✅ Saved: 01_model_diagnostics.html")

# ============================================================================
# 2. CROP-SPECIFIC ANALYSIS
# ============================================================================

# Crop Yield Distribution
fig_yield_dist = go.Figure()
unique_crops = df['crop'].unique()

for crop in unique_crops:
    crop_data = df[df['crop'] == crop]['yield_tonnes_per_acre']
    fig_yield_dist.add_trace(go.Box(
        y=crop_data, 
        name=str(crop),
        boxmean='sd'
    ))

fig_yield_dist.update_layout(
    title="Yield Distribution by Crop Type",
    yaxis_title="Yield (tonnes/acre)",
    height=500,
    showlegend=True
)
fig_yield_dist.write_html(f"{output_dir}/02_crop_yield_distribution.html")
print("✅ Saved: 02_crop_yield_distribution.html")

# Average Yield by Crop
crop_yield_avg = df.groupby('crop')['yield_tonnes_per_acre'].mean().reset_index()
crop_yield_avg.columns = ['Crop', 'Avg Yield']
crop_yield_avg = crop_yield_avg.sort_values('Avg Yield', ascending=True)

fig_crop_bar = px.bar(crop_yield_avg, 
    x='Avg Yield', 
    y='Crop',
    orientation='h',
    color='Avg Yield',
    color_continuous_scale='Viridis',
    title="Average Yield by Crop",
    labels={'Avg Yield': 'Average Yield (tonnes/acre)'},
    height=400
)
fig_crop_bar.write_html(f"{output_dir}/03_crop_avg_yield.html")
print("✅ Saved: 03_crop_avg_yield.html")

print(f"\n🌾 Crop Yield Statistics:")
print(crop_yield_avg.to_string(index=False))

# ============================================================================
# 3. WEATHER IMPACT ANALYSIS
# ============================================================================

# Yield vs Temperature by Province
fig_scatter = px.scatter(
    df,
    x='avg_temp',
    y='yield_tonnes_per_acre',
    color='province',
    size='area_acres',
    hover_name='crop',
    hover_data=['rainfall', 'area_acres'],
    title="Yield vs Temperature by Province (Size = Area)",
    labels={'avg_temp': 'Average Temperature (°C)', 
            'yield_tonnes_per_acre': 'Yield (tonnes/acre)'},
    height=600
)
fig_scatter.write_html(f"{output_dir}/04_yield_vs_temperature.html")
print("✅ Saved: 04_yield_vs_temperature.html")

# Rainfall impact visualization
unique_provinces = df['province'].unique()

fig_rainfall = make_subplots(
    rows=2, cols=2,
    subplot_titles=tuple(unique_provinces),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

for idx, province in enumerate(unique_provinces):
    row = idx // 2 + 1
    col = idx % 2 + 1
    prov_data = df[df['province'] == province]
    
    fig_rainfall.add_trace(
        go.Scatter(
            x=prov_data['rainfall'],
            y=prov_data['yield_tonnes_per_acre'],
            mode='markers',
            name=province,
            marker=dict(size=8, opacity=0.6)
        ),
        row=row, col=col
    )
    fig_rainfall.update_xaxes(title_text="Rainfall (mm)", row=row, col=col)
    fig_rainfall.update_yaxes(title_text="Yield (tonnes/acre)", row=row, col=col)

fig_rainfall.update_layout(height=700, title_text="Rainfall Impact on Yield by Province")
fig_rainfall.write_html(f"{output_dir}/05_rainfall_impact_provinces.html")
print("✅ Saved: 05_rainfall_impact_provinces.html")

print("\n📊 All visualizations completed and saved to frontend/static/reports/")


✅ Saved: 01_model_diagnostics.html
✅ Saved: 02_crop_yield_distribution.html
✅ Saved: 03_crop_avg_yield.html

🌾 Crop Yield Statistics:
  Crop  Avg Yield
Cotton    0.24625
 Maize    0.90372
 Wheat    0.98529
  Rice    1.02934
✅ Saved: 04_yield_vs_temperature.html
✅ Saved: 05_rainfall_impact_provinces.html

📊 All visualizations completed and saved to frontend/static/reports/
